# Figure 2 and Figure 3 panels

子图写到本目录的 `figures_fig2_fig3/`。页面规格与 `promoter/fig2.md`、`promoter/fig3.md` 一致。

- Figure 2 只用 fold 0：`best_mlp_gpu.pt` 是这一折的权重，一张切片整张在训练或验证里。
- Figure 3 用同一套权重在 H&E 上的预测。Spot 的颜色是落入该 spot 的细胞核 L2 概率的平均，不是另一个模型。

裁图和 fold 0 的分数表调用 `promoter/build_fig23.py`。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code"))
sys.path.insert(0, str(REPO / "code" / "Hist2Pheno_pkg"))
sys.path.insert(0, str(REPO / "promoter"))

import build_fig23 as fig23
from plotting_palettes import get_palette
from uni_label_cv_helpers import macro_auc_ovr

DATA = REPO / "data"
OUT = REPO / "code" / "Hist2Pheno_pkg" / "dataset" / "figures_fig2_fig3"
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "Liberation Sans",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.4,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

L2_PALETTE = get_palette("pancancer", "fine")
L12_PALETTE = get_palette("pancancer", "intermediate")
L1_PALETTE = get_palette("pancancer", "coarse")
TIER_PALETTE = {"L1": L1_PALETTE, "L12": L12_PALETTE, "L2": L2_PALETTE}

try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    def display(obj):
        print(obj)


def save_fig(fig, stem):
    png = OUT / f"{stem}.png"
    pdf = OUT / f"{stem}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(png)
    if Image is not None:
        display(Image(filename=str(png)))
    return png

## Figure 2a. Section is the hold-out group

每个格子是该验证折留出的整张切片数。灰色列是 fold 0。数字来自 `data/PanCancer/fold_plan.csv`。

In [ ]:
ORDER = ["Lung", "BRCA", "CRC", "HCC", "PDAC", "GIST", "GBM"]
SHOW = {"BRCA": "Breast"}

fold = pd.read_csv(DATA / "PanCancer/fold_plan.csv")
fold["fold"] = fold["fold"].astype(int)
counts = (
    fold.groupby(["dataset", "fold"]).size()
    .unstack("fold")
    .reindex(index=ORDER, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)
counts.to_csv(OUT / "fig2a_section_counts.csv")
print("fold 0 sections", int(counts[0].sum()))
display(counts.rename(index=SHOW))

fig, ax = plt.subplots(figsize=(8.6 / 2.54, 6.8 / 2.54))
ax.set_xlim(-1.35, 4.7)
ax.set_ylim(len(ORDER) - 0.45, -1.25)
ax.axis("off")
ax.text(2, -1.05, "Validation fold", ha="center", va="center", fontsize=7)
for j, fold_id in enumerate(range(5)):
    ax.text(j, -0.62, str(fold_id), ha="center", va="center", fontsize=7,
            fontweight="bold" if fold_id == 0 else "regular")
for i, dataset in enumerate(ORDER):
    ax.text(-0.55, i, SHOW.get(dataset, dataset), ha="right", va="center", fontsize=7)
    for j, fold_id in enumerate(range(5)):
        value = int(counts.loc[dataset, fold_id])
        if value == 0:
            continue
        ax.add_patch(plt.Rectangle(
            (j - 0.36, i - 0.34), 0.72, 0.68,
            facecolor="#E8E8E8" if fold_id == 0 else "white",
            edgecolor="black", linewidth=0.5, zorder=2,
        ))
        ax.text(j, i, str(value), ha="center", va="center", fontsize=7, zorder=3)
save_fig(fig, "fig2a_section_split")

## Figure 2a, continued. Kept cells and pathologist labels

上面那张图是切片数。下面两张用同一套 5 折。

- 细胞数是 `fold_plan.csv` 的 `n_keep`：映射到 pan-cancer 标签后、这一折留出的细胞核数。
- 条形图是这些细胞核的病理标注，存在 `data/PanCancer/label_cache/`。`y_l1`、`y_l12`、`y` 分别是 L1、L12、L2。它们来自各队列单独训练时的标注，经 `celltype_pancancer` 映射后保留。条形是该折、该癌种内部的比例。纵轴上加粗的 0 是 fold 0。某一折没有这个癌种时，这一行留空。

In [ ]:

CACHE = DATA / "PanCancer/label_cache"
cell_counts = (
    fold.groupby(["dataset", "fold"])["n_keep"].sum()
    .unstack("fold")
    .reindex(index=ORDER, columns=[0, 1, 2, 3, 4])
    .fillna(0)
    .astype(int)
)
cell_counts.to_csv(OUT / "fig2a_cell_counts.csv")
display(cell_counts.rename(index=SHOW))

fig, ax = plt.subplots(figsize=(14.2 / 2.54, 7.4 / 2.54))
ax.set_xlim(-1.7, 4.65)
ax.set_ylim(len(ORDER) - 0.45, -1.25)
ax.axis("off")
ax.text(2, -1.05, "Validation fold", ha="center", va="center", fontsize=7)
for j, fold_id in enumerate(range(5)):
    ax.text(j, -0.62, str(fold_id), ha="center", va="center", fontsize=7,
            fontweight="bold" if fold_id == 0 else "regular")
for i, dataset in enumerate(ORDER):
    ax.text(-0.62, i, SHOW.get(dataset, dataset), ha="right", va="center", fontsize=7)
    for j, fold_id in enumerate(range(5)):
        value = int(cell_counts.loc[dataset, fold_id])
        if value == 0:
            continue
        ax.add_patch(plt.Rectangle(
            (j - 0.46, i - 0.34), 0.92, 0.68,
            facecolor="#E8E8E8" if fold_id == 0 else "white",
            edgecolor="black", linewidth=0.5, zorder=2,
        ))
        ax.text(j, i, f"{value:,}", ha="center", va="center", fontsize=6, zorder=3)
save_fig(fig, "fig2a_cell_counts")

LEVEL_ARRAY = {"L1": "y_l1", "L12": "y_l12", "L2": "y"}
PALETTE = {"L1": L1_PALETTE, "L12": L12_PALETTE, "L2": L2_PALETTE}
from collections import Counter
gt_counts = {}
for rec in fold.itertuples(index=False):
    cache_path = CACHE / f"{rec.section_id.replace('::', '__')}.npz"
    z = np.load(cache_path, allow_pickle=True)
    for level, key in LEVEL_ARRAY.items():
        labels, n = np.unique(z[key], return_counts=True)
        bucket = gt_counts.setdefault((rec.dataset, int(rec.fold), level), Counter())
        for label, count in zip(labels, n):
            bucket[str(label)] += int(count)

rows = []
for (dataset, fold_id, level), bucket in gt_counts.items():
    total = sum(bucket.values())
    for label, count in bucket.items():
        rows.append({
            "dataset": dataset,
            "fold": fold_id,
            "level": level,
            "label": label,
            "n": count,
            "fraction": count / total,
        })
gt_table = pd.DataFrame(rows).sort_values(["level", "dataset", "fold", "label"])
gt_table.to_csv(OUT / "fig2a_gt_composition.csv", index=False)
print(f"kept cells in cache {int(gt_table.loc[gt_table.level == 'L2', 'n'].sum()):,}")

fig, axes = plt.subplots(
    len(ORDER), 3,
    figsize=(18.0 / 2.54, 16.2 / 2.54),
    sharex=True,
    gridspec_kw={"wspace": 0.18, "hspace": 0.28},
)
for i, dataset in enumerate(ORDER):
    for j, level in enumerate(("L1", "L12", "L2")):
        ax = axes[i, j]
        ax.axhspan(-0.46, 0.46, color="#F3F3F3", zorder=0)
        palette = PALETTE[level]
        present = [
            name for name in palette
            if any(gt_counts.get((dataset, fold_id, level), {}).get(name, 0) for fold_id in range(5))
        ]
        for fold_id in range(5):
            bucket = gt_counts.get((dataset, fold_id, level), {})
            total = sum(bucket.values())
            if total == 0:
                continue
            left = 0.0
            for name in present:
                count = bucket.get(name, 0)
                if count == 0:
                    continue
                width = count / total
                ax.barh(
                    fold_id, width, left=left, height=0.72,
                    color=palette[name], linewidth=0, zorder=2,
                )
                left += width
        ax.set_xlim(0, 1)
        ax.set_ylim(4.55, -0.55)
        ax.tick_params(length=2, width=0.4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if j == 0:
            ax.set_yticks(range(5))
            ax.set_yticklabels([str(k) for k in range(5)])
            for tick in ax.get_yticklabels():
                if tick.get_text() == "0":
                    tick.set_fontweight("bold")
            ax.set_ylabel(SHOW.get(dataset, dataset), fontsize=7)
        else:
            ax.set_yticks(range(5))
            ax.set_yticklabels([])
        if i == 0:
            ax.set_title(level, fontsize=7, pad=3)
        if i == len(ORDER) - 1:
            ax.set_xlabel("Fraction", fontsize=7)
        else:
            ax.set_xlabel("")
            ax.tick_params(labelbottom=False)

for j, level in enumerate(("L1", "L12", "L2")):
    palette = PALETTE[level]
    used = {
        row.label
        for row in gt_table.itertuples(index=False)
        if row.level == level
    }
    present = [name for name in palette if name in used]
    handles = [
        plt.Line2D(
            [0], [0], linestyle="none", marker="s", markersize=4,
            markerfacecolor=palette[name], markeredgecolor="none", label=name,
        )
        for name in present
    ]
    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.20 + 0.30 * j, -0.01),
        frameon=False,
        fontsize=5.5,
        ncol=2 if level == "L1" else 3,
        columnspacing=0.6,
        handletextpad=0.25,
    )
fig.savefig(OUT / "fig2a_gt_composition.png", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(OUT / "fig2a_gt_composition.pdf", bbox_inches="tight", facecolor="white")
plt.close(fig)
print(OUT / "fig2a_gt_composition.png")
if Image is not None:
    display(Image(filename=str(OUT / "fig2a_gt_composition.png")))


## Figure 2b. Held-out macro-AUROC

每个点是 fold 0 的一张切片。`fold0_scores` 把 `metrics_per_section.csv` 接到 `fold_plan.csv`，只保留 `fold == 0`。LungIncomplete 和 CRC `P5CRC` 不在这张图里。黑线是中位数。

In [ ]:
series = fig23.fold0_scores()
rows = []
for name, levels in series:
    n = len(levels["L2"])
    for level, values in levels.items():
        for value in values:
            rows.append({"cancer": name, "level": level, "macro_auroc": value, "n_sections": n})
points = pd.DataFrame(rows)
points.to_csv(OUT / "fig2b_fold0_macro_auroc.csv", index=False)
summary = (
    points.groupby(["cancer", "level"])["macro_auroc"]
    .agg(n="size", median="median")
    .reset_index()
)
display(summary.pivot(index="cancer", columns="level", values="median").reindex(
    [name for name, _ in series]
)[["L1", "L12", "L2"]])

fig, ax = plt.subplots(figsize=(11.2 / 2.54, 6.2 / 2.54))
rng = np.random.default_rng(0)
for i, (name, levels) in enumerate(series):
    for j, level in enumerate(("L1", "L12", "L2")):
        vals = np.asarray(levels[level], dtype=float)
        xx = i + (j - 1) * 0.22
        color = fig23.LEVEL_COLOR[level]
        if len(vals) == 1:
            ax.scatter([xx], vals, s=18, c=color, zorder=3, label=level if i == 0 else None)
        else:
            jitter = rng.uniform(-0.05, 0.05, size=len(vals))
            ax.scatter(
                np.full(len(vals), xx) + jitter, vals,
                s=9, c=color, alpha=0.55, linewidths=0, zorder=3,
                label=level if i == 0 else None,
            )
            med = float(np.median(vals))
            ax.plot([xx - 0.08, xx + 0.08], [med, med], color="black", lw=0.8, zorder=4)
ax.set_xticks(range(len(series)))
ax.set_xticklabels([name for name, _ in series])
ax.set_ylabel("Macro-AUROC")
ax.set_xlim(-0.6, len(series) - 0.4)
ax.set_ylim(0.45, 1.02)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18), borderaxespad=0)
save_fig(fig, "fig2b_fold0_macro_auroc")

## Figure 2c. Held-out HCC section

`kqr-83230_aligned_3bba12d6` 在 fold 0。接触表 PNG 早于现在的 `plotting_palettes.py`，所以这里不裁那张图，而是用 `matched_pred.h5ad` 重画。颜色是 `get_palette("pancancer")`：L1 coarse、L12 intermediate、L2 fine。每一列最下方是该列在标注或预测里出现过的类。

In [ ]:
import anndata as ad
from matplotlib.lines import Line2D

hcc_h5 = (
    DATA / "PanCancer/pred_all_spatial/matched/HCC/kqr-83230_aligned_3bba12d6"
    / "kqr-83230_aligned_3bba12d6_matched_pred.h5ad"
)
adata = ad.read_h5ad(hcc_h5)
xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
columns = [
    ("L1", "final_lineage", "pred_lineage", L1_PALETTE),
    ("L12", "final_sublineage", "pred_sublineage", L12_PALETTE),
    ("L2", "final_CT", "pred_CT", L2_PALETTE),
]

def scatter_labels(ax, labels, palette):
    labels = np.asarray(labels, dtype=str)
    for name, color in palette.items():
        mask = labels == name
        if np.any(mask):
            ax.scatter(
                xy[mask, 0], xy[mask, 1], s=0.4, c=[color],
                linewidths=0, rasterized=True,
            )
    ax.set_xlim(xy[:, 0].min(), xy[:, 0].max())
    ax.set_ylim(xy[:, 1].max(), xy[:, 1].min())
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])

fig = plt.figure(figsize=(16.4 / 2.54, 12.6 / 2.54))
gs = fig.add_gridspec(3, 3, height_ratios=[1.0, 1.0, 0.62], hspace=0.08, wspace=0.12)
for j, (head, gt_col, pred_col, palette) in enumerate(columns):
    for i, (col, row_name) in enumerate(((gt_col, "Ground truth"), (pred_col, "Hist2Pheno"))):
        ax = fig.add_subplot(gs[i, j])
        scatter_labels(ax, adata.obs[col].to_numpy(), palette)
        if i == 0:
            ax.set_title(head, fontsize=7, pad=2)
        if j == 0:
            ax.set_ylabel(row_name, fontsize=7)
    seen = set(adata.obs[gt_col].astype(str)) | set(adata.obs[pred_col].astype(str))
    present = [name for name in palette if name in seen]
    axL = fig.add_subplot(gs[2, j])
    axL.axis("off")
    handles = [
        Line2D(
            [0], [0], linestyle="none", marker="o", markersize=4,
            markerfacecolor=palette[name], markeredgecolor="none", label=name,
        )
        for name in present
    ]
    axL.legend(
        handles=handles, loc="upper center", frameon=False, fontsize=5.5,
        ncol=2 if len(present) > 4 else 1,
        columnspacing=0.6, handletextpad=0.25, borderaxespad=0.0,
    )
    print(head, present)

save_fig(fig, "fig2c_hcc_gt_pred")

## Figure 2d. Class-wise ROC on that section

`stardist_roc/` 里的 CSV 是逐核的预测和概率，不是已经算好的 AUC。这里按 `macro_auc_ovr` 的规则重画 one-versus-rest：某一类在该切片上没有阳性或没有阴性就不画。

In [ ]:
ROC_DIR = DATA / "PanCancer/pred_all_spatial/stardist_roc/HCC__kqr-83230_aligned_3bba12d6"

def load_tier(pred_name, names_name, name_col):
    pred = pd.read_csv(ROC_DIR / pred_name)
    names = pd.read_csv(ROC_DIR / names_name).sort_values("class_index")
    classes = names[name_col].astype(str).tolist()
    prob_cols = sorted(
        [c for c in pred.columns if c.startswith("prob_")],
        key=lambda c: int(c.split("_")[1]),
    )
    y = pred["ground_truth_idx"].to_numpy(dtype=np.int64)
    score = pred[prob_cols].to_numpy(dtype=np.float64)
    if score.shape[1] != len(classes):
        raise RuntimeError(f"{pred_name}: {score.shape[1]} probability columns, {len(classes)} classes")
    return classes, y, score

from sklearn.metrics import auc, roc_curve
from sklearn.preprocessing import label_binarize

tiers = {
    "L1": load_tier(
        "validation_external_stardist_matched_AUROC_level1.csv",
        "validation_external_stardist_matched_AUROC_level1_class_names.csv",
        "final_lineage",
    ),
    "L12": load_tier(
        "validation_external_stardist_matched_AUROC_level12.csv",
        "validation_external_stardist_matched_AUROC_level12_class_names.csv",
        "final_sublineage",
    ),
    "L2": load_tier(
        "validation_external_stardist_matched_AUROC.csv",
        "validation_external_stardist_matched_AUROC_class_names.csv",
        "final_CT",
    ),
}

fig, axes = plt.subplots(1, 3, figsize=(16.2 / 2.54, 5.6 / 2.54))
auc_rows = []
for ax, level in zip(axes, ("L1", "L12", "L2")):
    classes, y, score = tiers[level]
    macro = macro_auc_ovr(y, score, len(classes))
    Y = label_binarize(y, classes=np.arange(len(classes)))
    if Y.ndim == 1:
        Y = np.column_stack([1 - Y, Y])
    palette = TIER_PALETTE[level]
    for k, name in enumerate(classes):
        pos = int(Y[:, k].sum())
        if pos == 0 or pos == len(Y):
            continue
        fpr, tpr, _ = roc_curve(Y[:, k], score[:, k])
        class_auc = float(auc(fpr, tpr))
        ax.plot(fpr, tpr, color=palette.get(name, (0.4, 0.4, 0.4, 1)), lw=0.8, label=f"{name}  {class_auc:.2f}")
        auc_rows.append({"level": level, "class": name, "auroc": class_auc, "n_positive": pos})
    ax.plot([0, 1], [0, 1], color="0.6", lw=0.4, ls="--")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title(f"{level}   macro {macro:.3f}")
    ax.set_aspect("equal", adjustable="box")
    if level == "L2":
        ax.legend(frameon=False, fontsize=5, loc="center left", bbox_to_anchor=(1.02, 0.5))
    else:
        ax.legend(frameon=False, fontsize=5, loc="lower right")
    print(f"{level} macro-AUROC {macro:.3f}")

pd.DataFrame(auc_rows).to_csv(OUT / "fig2d_hcc_per_class_auroc.csv", index=False)
save_fig(fig, "fig2d_hcc_roc")

## Figure 3. Fields already on disk

每一行四格的顺序是 H&E、细胞核 L2、spot、一个 niche。这一格只导出已经画好的 H&E、L2 和 niche。

- L2 取空间图的右栏。
- Niche 取三栏图的中间栏，也就是半径 55 µm 的空间得分。
- HCC 用 `P1_N` 和 `P1_T` 的 SRI。
- GBM 用 `P174511_Initial` 接触表最下面一行的 L2，也就是全部 StarDist 核。
- CRC 的 H&E 是 11 GB BigTIFF，这里不解码。

In [ ]:
def export_he(src, dest):
    if src is None or not Path(src).is_file():
        print("H&E left blank:", dest.name)
        return None
    fig23.he_thumb(src, dest, max_px=900)
    print(dest)
    return dest

fields = []

def add_field(letter, cancer, kind, path):
    fields.append({"letter": letter, "cancer": cancer, "field": kind, "path": str(path) if path else ""})

specs = [
    ("a", "Prostate",
     DATA / "Visium/Prostate/Patient_1/HE_images/H1_2.jpg",
     DATA / "Visium/Prostate/pred_pancancer_section/figures/Patient_1__H1_2_spatial_L1_L12_L2.png",
     DATA / "Visium/Prostate/pred_pancancer_section/figures/Patient_1__H1_2_TLS_abundance_vs_spatial.png",
     "tls"),
    ("b", "NPC",
     DATA / "Visium/NPC/HE_images/20210809-C-AH4199551.tif",
     DATA / "Visium/NPC/pred_pancancer_section/figures/20210809-C-AH4199551_spatial_L1_L12_L2.png",
     DATA / "Visium/NPC/pred_pancancer_section/figures/20210809-C-AH4199551_TLS_abundance_vs_spatial.png",
     "tls"),
    ("c_N", "HCC P1_N",
     DATA / "Visium/HCC/HE_images/P1_N.jpg",
     DATA / "Visium/HCC/pred_pancancer_section/figures/P1_N_spatial_L1_L12_L2.png",
     DATA / "Visium/HCC/pred_pancancer_section/figures/P1_N_SRI_abundance_vs_spatial.png",
     "sri"),
    ("c_T", "HCC P1_T",
     DATA / "Visium/HCC/HE_images/P1_T.jpg",
     DATA / "Visium/HCC/pred_pancancer_section/figures/P1_T_spatial_L1_L12_L2.png",
     DATA / "Visium/HCC/pred_pancancer_section/figures/P1_T_SRI_abundance_vs_spatial.png",
     "sri"),
    ("d", "Breast",
     DATA / "ST/BRCA/HE_images/10426_HE_BT_C1.jpg",
     DATA / "ST/BRCA/pred_pancancer_section/figures/10426_HE_BT_C1_spatial_L1_L12_L2.png",
     DATA / "ST/BRCA/pred_pancancer_section/figures/10426_HE_BT_C1_TLS_abundance_vs_spatial.png",
     "tls"),
    ("e", "CRC",
     None,
     DATA / "Visium/CRC/pred_pancancer_section/figures/Visium_V2_Human_Colon_Cancer_P2_tissue_image_spatial_L1_L12_L2.png",
     DATA / "Visium/CRC/pred_pancancer_section/figures/Visium_V2_Human_Colon_Cancer_P2_tissue_image_TLS_abundance_vs_spatial.png",
     "tls"),
]

for letter, cancer, he, spatial, niche, niche_name in specs:
    he_out = OUT / f"fig3{letter}_he.jpg"
    l2_out = OUT / f"fig3{letter}_l2.png"
    niche_out = OUT / f"fig3{letter}_{niche_name}.png"
    export_he(he, he_out)
    fig23.l2_third(spatial, l2_out)
    fig23.niche_middle(niche, niche_out)
    print(l2_out)
    print(niche_out)
    add_field(letter, cancer, "H&E", he_out if he else None)
    add_field(letter, cancer, "L2", l2_out)
    add_field(letter, cancer, niche_name, niche_out)

gbm_he = (
    DATA / "CODEX/GBM/WangLab/4_Images/CytAssist_Image"
    / "P174511_2_3_2024-05-11_14-33-44_2024-05-11_14-05-32_H1-8BZDCBG_A1_sample.tif"
)
gbm_panel = (
    DATA / "PanCancer/pred_all_spatial/matched/GBM/P174511_Initial"
    / "P174511_Initial_panel_4x3.png"
)
export_he(gbm_he, OUT / "fig3f_he.jpg")
fig23.gbm_l2(gbm_panel, OUT / "fig3f_l2.png")
print(OUT / "fig3f_l2.png")
add_field("f", "GBM", "H&E", OUT / "fig3f_he.jpg")
add_field("f", "GBM", "L2", OUT / "fig3f_l2.png")
print("CRC H&E source is Visium_V2_Human_Colon_Cancer_P2_tissue_image.btf (11 GB). Drop a JPEG into this folder when it is ready.")

## Figure 3 spot composition

一个 spot 取落在它里面的细胞核，对 L2 概率求平均，再按平均概率最大的那一类上色。

- Breast `10426_HE_BT_C1` 对应 HER2ST `D1`。坐标在 `ST-spotfiles/D1_selection.tsv` 的 `pixel_x`、`pixel_y`，与 `spatial_HE` 同一像素系。分配半径是 spot 间距的一半。概率用 `10426_HE_BT_C1_l2_probs.npz`。名义间距 200 µm。
- GBM `P174511_Initial` 用 16 µm bin。`tissue_positions.parquet` 里 `in_tissue == 1` 的 bin，半径是 bin 间距的一半。这张切片的 StarDist h5ad 只存了 `pred_CT`，没有 softmax，所以 bin 里记的是预测标签的比例，颜色是多数类。
- Prostate、NPC、HCC、CRC 的 spot 坐标不在各自的 `data/` 目录里，这四格不画。

In [ ]:
import json
import anndata as ad
from matplotlib.patches import Circle
from scipy.spatial import cKDTree

L2_NAMES = list(L2_PALETTE)


def nearest_within(nuc_xy, spot_xy, radius_px):
    dist, index = cKDTree(spot_xy).query(nuc_xy, k=1)
    keep = dist <= radius_px
    return index, keep


def mean_by_group(index, keep, values, n_groups):
    sums = np.zeros((n_groups, values.shape[1]), dtype=np.float64)
    counts = np.zeros(n_groups, dtype=np.int32)
    np.add.at(sums, index[keep], values[keep])
    np.add.at(counts, index[keep], 1)
    mean = np.divide(sums, counts[:, None], out=np.zeros_like(sums), where=counts[:, None] > 0)
    return mean, counts


def one_hot(labels, names):
    lookup = {name: i for i, name in enumerate(names)}
    missing = sorted(set(map(str, labels)) - set(lookup))
    if missing:
        raise RuntimeError("labels outside the L2 palette: " + ", ".join(missing))
    idx = np.fromiter((lookup[str(x)] for x in labels), dtype=np.int64, count=len(labels))
    return np.eye(len(names), dtype=np.float32)[idx]


def draw_spots(xy, dominant, pitch_px, title, stem, counts):
    keep = counts > 0
    xy = np.asarray(xy)[keep]
    dominant = np.asarray(dominant)[keep]
    fig, ax = plt.subplots(figsize=(7.4 / 2.54, 7.2 / 2.54))
    present = [name for name in L2_NAMES if np.any(dominant == name)]
    if len(xy) > 4000:
        for name in present:
            mask = dominant == name
            ax.scatter(
                xy[mask, 0], xy[mask, 1],
                s=2, c=[L2_PALETTE[name]], linewidths=0, rasterized=True,
            )
    else:
        patches, colors = [], []
        radius = 0.45 * float(pitch_px)
        for name in present:
            for x, y in xy[dominant == name]:
                patches.append(Circle((x, y), radius))
                colors.append(L2_PALETTE[name])
        ax.add_collection(plt.matplotlib.collections.PatchCollection(
            patches, facecolors=colors, edgecolors="none",
        ))
    pad = float(pitch_px)
    ax.set_xlim(xy[:, 0].min() - pad, xy[:, 0].max() + pad)
    ax.set_ylim(xy[:, 1].max() + pad, xy[:, 1].min() - pad)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    handles = [
        plt.Line2D([0], [0], marker="o", color="none", markerfacecolor=L2_PALETTE[name],
                   markeredgecolor="none", markersize=4, label=name)
        for name in present
    ]
    ax.legend(handles=handles, frameon=False, fontsize=5, loc="center left", bbox_to_anchor=(1.02, 0.5))
    save_fig(fig, stem)
    return present


# --- Breast, HER2ST D1 ---
sample = "10426_HE_BT_C1"
pred_path = DATA / "ST/BRCA/pred_pancancer_section" / sample / f"{sample}_stardist_pred.h5ad"
prob_path = pred_path.with_name(f"{sample}_l2_probs.npz")
spot_path = DATA / "ST/BRCA/her2st/data/ST-spotfiles/D1_selection.tsv"

adata = ad.read_h5ad(pred_path)
nuc_xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
z = np.load(prob_path, allow_pickle=True)
probs = np.asarray(z["probs"], dtype=np.float32)
class_names = [str(x) for x in z["class_names"].tolist()]
if set(class_names) != set(L2_NAMES):
    raise RuntimeError("L2 names in the npz are not the pan-cancer palette")
if probs.shape[0] != nuc_xy.shape[0]:
    raise RuntimeError(f"probs {probs.shape[0]} != nuclei {nuc_xy.shape[0]}")

spots = pd.read_csv(spot_path, sep="\t")
spots = spots.loc[spots["selected"] == 1].reset_index(drop=True)
spot_xy = spots[["pixel_x", "pixel_y"]].to_numpy(dtype=np.float64)
pitch_px = float(np.median(cKDTree(spot_xy).query(spot_xy, k=2)[0][:, 1]))
radius_px = 0.5 * pitch_px
index, keep = nearest_within(nuc_xy, spot_xy, radius_px)
mean, counts = mean_by_group(index, keep, probs, len(spot_xy))
dominant = np.array(class_names)[mean.argmax(1)]
dominant = np.where(counts > 0, dominant, "")

breast = spots.copy()
breast["n_nuclei"] = counts
breast["dominant_l2"] = dominant
breast["dominant_mean_prob"] = np.where(counts > 0, mean.max(1), np.nan)
for i, name in enumerate(class_names):
    breast[f"mean_{name}"] = mean[:, i]
breast.to_csv(OUT / "fig3d_breast_spot_composition.csv", index=False)
print(
    f"Breast spots {len(spot_xy)}, pitch {pitch_px:.1f} px, radius {radius_px:.1f} px, "
    f"nuclei assigned {int(keep.sum())} / {len(nuc_xy)}, spots with nuclei {(counts > 0).sum()}"
)
draw_spots(
    spot_xy, dominant, pitch_px,
    "HER2ST D1    200 µm spots    mean L2 probability",
    "fig3d_breast_spot_l2",
    counts,
)

# --- GBM, 16 µm bins. Hard labels, because softmax was not stored. ---
gbm_h5 = (
    DATA / "PanCancer/pred_all_spatial/stardist_all/GBM/P174511_Initial"
    / "P174511_Initial_all_stardist_pred.h5ad"
)
pos_path = (
    DATA / "CODEX/GBM/WangLab/1_Bin_Matrix/P174511_Initial/square_016um/spatial/tissue_positions.parquet"
)
scale_path = pos_path.with_name("scalefactors_json.json")
scale = json.loads(scale_path.read_text())
bin_px = float(scale["spot_diameter_fullres"])
radius_px = 0.5 * bin_px

gbm = ad.read_h5ad(gbm_h5)
nuc_xy = np.asarray(gbm.obsm["spatial_HE"], dtype=np.float64)
hard = one_hot(gbm.obs["pred_CT"].astype(str).to_numpy(), L2_NAMES)
pos = pd.read_parquet(pos_path)
pos = pos.loc[pos["in_tissue"] == 1].reset_index(drop=True)
bin_xy = pos[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy(dtype=np.float64)
index, keep = nearest_within(nuc_xy, bin_xy, radius_px)
mean, counts = mean_by_group(index, keep, hard, len(bin_xy))
dominant = np.array(L2_NAMES)[mean.argmax(1)]
dominant = np.where(counts > 0, dominant, "")

used = counts > 0
gbm_out = pd.DataFrame({
    "barcode": pos.loc[used, "barcode"].to_numpy(),
    "pixel_x": bin_xy[used, 0],
    "pixel_y": bin_xy[used, 1],
    "n_nuclei": counts[used],
    "dominant_l2": dominant[used],
    "dominant_fraction": mean.max(1)[used],
})
gbm_out.to_csv(OUT / "fig3f_gbm_bin16_composition.csv", index=False)
print(
    f"GBM bins in tissue {len(bin_xy)}, pitch {bin_px:.2f} px ({scale['bin_size_um']} µm), "
    f"nuclei inside a bin {int(keep.sum())} / {len(nuc_xy)}, bins with nuclei {int(used.sum())}"
)
draw_spots(
    bin_xy, dominant, bin_px,
    "P174511_Initial    16 µm bins    majority L2 label",
    "fig3f_gbm_bin16_l2",
    counts,
)

skipped = pd.DataFrame([
    {"letter": "a", "cancer": "Prostate", "reason": "No tissue_positions in data/Visium/Prostate"},
    {"letter": "b", "cancer": "NPC", "reason": "No tissue_positions in data/Visium/NPC"},
    {"letter": "c", "cancer": "HCC", "reason": "No spatial/ directory for the P1_N and P1_T JPEGs"},
    {"letter": "e", "cancer": "CRC", "reason": "No tissue_positions in data/Visium/CRC"},
])
skipped.to_csv(OUT / "fig3_spot_maps_not_drawn.csv", index=False)
display(skipped)

## What was written

上色用的是 `plotting_palettes.get_palette("pancancer")`。这台机器没有 Arial，图里用 Liberation Sans，字号按 7 pt。

In [ ]:
written = sorted(p.name for p in OUT.iterdir() if p.is_file())
print(f"{len(written)} files in {OUT}")
for name in written:
    print(" ", name)